# 02 - Comparative experiments

Config is pinned step by step: embedder first, then LLM, then prompt. At each step the best candidate is kept and used as the base for the next one.

- Study A: 3 embedders (baseline LLM + strict prompt)
- Study B: 3 LLMs (best embedder + strict prompt)
- Study C: 3 prompts (best embedder + best LLM)

In [1]:
import sys, gc, torch
sys.path.insert(0, '..')

from src.embedding import Embedder
from src.vectorstore import VectorStore
from src.generation import LLMGenerator
from src.rag_pipeline import RAGPipeline
from evaluation.evaluate import (
    load_dataset, retrieval_recall, keyword_score, measure_latency
)
import pandas as pd

dataset = load_dataset('../evaluation/qa_dataset.json')
print('nb questions:', len(dataset))

nb questions: 20


In [2]:
def free_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        torch.mps.empty_cache()

def unload(obj):
    m = getattr(obj, 'model', None)
    if m is not None and hasattr(m, 'to'):
        try:
            m.to('cpu')
        except Exception:
            pass


## Study A - embedders

In [3]:
CONFIGS_EMB = [
    ('minilm', 'sentence-transformers/all-MiniLM-L6-v2', 'collection_minilm'),
    ('e5_base', 'intfloat/multilingual-e5-base', 'collection_e5_base'),
    ('solon', 'OrdalieTech/Solon-embeddings-base-0.1', 'collection_solon_base'),
]

print('loading baseline llm (qwen 1.5b)...')
llm = LLMGenerator('Qwen/Qwen2.5-1.5B-Instruct')

results_A = []
for name, model, coll in CONFIGS_EMB:
    print(f'\n--- {name} ---')
    emb = Embedder(model)
    vs = VectorStore('../data/chroma_db', coll)
    pipe = RAGPipeline(emb, vs, llm, 'strict')
    r = retrieval_recall(pipe, dataset, k=4)
    k_score, _ = keyword_score(pipe, dataset, k=4)
    lat = measure_latency(pipe, dataset, k=4)
    results_A.append({
        'embedder': name,
        'recall@4': round(r, 3),
        'keyword': round(k_score, 3),
        'gen_ms': round(lat['generation']['median'], 0),
    })
    print(results_A[-1])
    unload(emb)
    del emb, vs, pipe
    free_memory()

df_A = pd.DataFrame(results_A)
print(df_A)
df_A.to_csv('../evaluation/results_A_embedders.csv', index=False)


loading baseline llm (qwen 1.5b)...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


--- minilm ---


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'embedder': 'e5_base', 'recall@4': 0.611, 'keyword': 0.191, 'gen_ms': 6745.0}

--- solon ---


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

{'embedder': 'solon', 'recall@4': 0.611, 'keyword': 0.191, 'gen_ms': 3408.0}
  embedder  recall@4  keyword  gen_ms
0   minilm     0.667    0.126  1936.0
1  e5_base     0.611    0.191  6745.0
2    solon     0.611    0.191  3408.0


## Study B - LLMs

In [4]:
best_row = df_A.sort_values('recall@4', ascending=False).iloc[0]
best_emb_name = best_row['embedder']
name_to_cfg = {n: (m, c) for (n, m, c) in CONFIGS_EMB}
best_model, best_coll = name_to_cfg[best_emb_name]
print('best embedder:', best_emb_name)

try:
    unload(llm)
    del llm
except NameError:
    pass
free_memory()

emb = Embedder(best_model)
vs = VectorStore('../data/chroma_db', best_coll)

CONFIGS_LLM = [
    ('tinyllama_1b', 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'),
    ('qwen_1.5b', 'Qwen/Qwen2.5-1.5B-Instruct'),
    ('qwen_3b', 'Qwen/Qwen2.5-3B-Instruct'),
]

results_B = []
llm = None
for name, model in CONFIGS_LLM:
    print(f'\n--- {name} ---')
    if llm is not None:
        unload(llm)
        del llm
        llm = None
    free_memory()
    try:
        llm = LLMGenerator(model)
    except Exception as e:
        print(f'skip {name}: {e}')
        continue
    pipe = RAGPipeline(emb, vs, llm, 'strict')
    k_score, _ = keyword_score(pipe, dataset, k=4)
    lat = measure_latency(pipe, dataset, k=4)
    results_B.append({
        'llm': name,
        'keyword': round(k_score, 3),
        'gen_ms': round(lat['generation']['median'], 0),
    })
    print(results_B[-1])

df_B = pd.DataFrame(results_B)
print(df_B)
df_B.to_csv('../evaluation/results_B_llms.csv', index=False)


best embedder: minilm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- tinyllama_1b ---


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

{'llm': 'tinyllama_1b', 'keyword': 0.062, 'gen_ms': 21980.0}

--- qwen_1.5b ---


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

{'llm': 'qwen_1.5b', 'keyword': 0.106, 'gen_ms': 2268.0}

--- qwen_3b ---


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

{'llm': 'qwen_3b', 'keyword': 0.183, 'gen_ms': 3855.0}
            llm  keyword   gen_ms
0  tinyllama_1b    0.062  21980.0
1     qwen_1.5b    0.106   2268.0
2       qwen_3b    0.183   3855.0


## Study C - prompts

In [5]:
best_llm_row = df_B.sort_values('keyword', ascending=False).iloc[0]
best_llm_name = best_llm_row['llm']
name_to_llm = {n: m for (n, m) in CONFIGS_LLM}
best_llm_model = name_to_llm[best_llm_name]
print('best llm:', best_llm_name)

try:
    unload(llm)
    del llm
except NameError:
    pass
free_memory()
llm = LLMGenerator(best_llm_model)

results_C = []
examples_c = []
for prompt_name in ['strict', 'citation', 'structured']:
    print(f'\n--- prompt {prompt_name} ---')
    pipe = RAGPipeline(emb, vs, llm, prompt_name)
    k_score, details = keyword_score(pipe, dataset, k=4)
    results_C.append({'prompt': prompt_name, 'keyword': round(k_score, 3)})
    print(results_C[-1])
    for d in details[:2]:
        examples_c.append({'prompt': prompt_name, **d})

df_C = pd.DataFrame(results_C)
print(df_C)
df_C.to_csv('../evaluation/results_C_prompts.csv', index=False)
pd.DataFrame(examples_c).to_csv('../evaluation/examples_qualitative.csv', index=False)

best llm: qwen_3b


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]


--- prompt strict ---
{'prompt': 'strict', 'keyword': 0.183}

--- prompt citation ---
{'prompt': 'citation', 'keyword': 0.247}

--- prompt structured ---
{'prompt': 'structured', 'keyword': 0.192}
       prompt  keyword
0      strict    0.183
1    citation    0.247
2  structured    0.192


## Retained config

In [6]:
best_prompt = df_C.sort_values('keyword', ascending=False).iloc[0]['prompt']
print('retained config:')
print(f'  embedder: {best_emb_name}')
print(f'  llm: {best_llm_name}')
print(f'  prompt: {best_prompt}')


retained config:
  embedder: minilm
  llm: qwen_3b
  prompt: citation

update app.py with these values before the demo.
